# Exploring FHIR R4 Bundles with pandas

This notebook loads FHIR Bundles from a local folder or S3, turns them into one pandas DataFrame per resource type, and shows example queries.

**Using a notebook:** run cells one at a time with **Shift+Enter**, or use *Run → Run All Cells*. Cells share state, so run them in order from the top. If things get confusing, use *Kernel → Restart Kernel and Run All Cells*.

Change `SOURCE` in the **Parameters** cell to point at your data.

In [1]:
# Make sure the cup_fhir package (in ../src) and its dependencies are available.
# On a fresh clone (e.g. Seqera Studios, JupyterHub) this installs them into the notebook's
# own Python the first time it runs. It does nothing once cup_fhir is installed.
import importlib.util, sys
from pathlib import Path

if importlib.util.find_spec("cup_fhir") is None:
    repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    %pip install --quiet -e "{repo_root}"
    # A new install is only picked up automatically after a kernel restart, so add it for this session too
    sys.path.insert(0, str(repo_root / "src"))

In [2]:
# Reload cup_fhir automatically when its source files change (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import pandas as pd

from cup_fhir import (
    SNOMED,
    amt_lookup, amt_search,
    build_vcf_table,
    extract_med_ingredients, med_ingredients_to_ctpp,
    has_code, with_patient,
    load_bundles, validate_bundles, to_dataframes,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

## Parameters

In [3]:
# Where to load from: an s3:// URI or a local folder (relative to this notebook)
SOURCE = "../data/real"
# SOURCE = "../data/sample"
# SOURCE = "s3://my-bucket/path/to/fhir/"

## 1. Load the bundles

`load_bundles` scans `SOURCE` for `.json`, `.ndjson` and gzipped files and keeps everything whose `resourceType` is `Bundle`. Problem files are reported rather than stopping the load.

In [4]:
result = load_bundles(SOURCE)
result

LoadResult(6 bundles, 0 files without bundles, 0 errors)

## 2. One DataFrame per resource type

Nested objects become dotted columns (`subject.reference`, `valueQuantity.value`). Arrays such as `code.coding`, `name` and `identifier` stay as list columns, and the `cup_fhir` helpers query those.

Each row also gets `_source` (which Bundle it came from) and `_fullUrl` (needed to resolve references).

In [5]:
dfs = to_dataframes(result.bundles)

patients              = dfs["Patient"].copy()
conditions            = dfs.get("Condition", pd.DataFrame())
med_administrations   = dfs.get("MedicationAdministration", pd.DataFrame())

patients["study_id"] = patients["identifier"].apply(
    lambda ids: ids[0]["value"] if isinstance(ids, list) and ids else None
)

{resource_type: len(df) for resource_type, df in dfs.items()}

{'Condition': 6,
 'DocumentReference': 30,
 'MedicationAdministration': 2,
 'Patient': 6,
 'Specimen': 24}

## 3. Patients

In [6]:
patients[["study_id"]]

,study_id
0,SN1
1,SN10
2,SN30
3,SN35
4,SN36
5,SN39


In [7]:
# Look up a patient by study ID
patients[patients["study_id"] == "SN30"]

,_source,_fullUrl,resourceType,identifier,text.status,text.div,study_id
2,../data/real/SN30.json#0,urn:uuid:17e5f6f0-dce4-55b7-894d-5c796fefa71a,Patient,[{'value': 'SN30'}],generated,"<div xmlns=""http://www.w3.org/1999/xhtml""><p><b>Patient</b></p><p>Paper ID: ...",SN30


## 4. Conditions

In [8]:
# Diagnosis codes present in this dataset
conditions["code.coding"].apply(
    lambda c: c[0].get("display") if isinstance(c, list) and c else None
).value_counts()

code.coding
Malignant neoplasm of primary unknown site    6
Name: count, dtype: int64

In [9]:
# Filter by SNOMED code, e.g. 399068003 = Malignant neoplasm of primary unknown site (CUP)
cup_mask = has_code(conditions["code.coding"], "399068003", SNOMED)
cup = with_patient(conditions[cup_mask], patients, columns=("study_id",))
cup[["patient_study_id", "onsetAge.value", "onsetAge.unit"]]

,patient_study_id,onsetAge.value,onsetAge.unit
0,SN1,67,years
1,SN10,56,years
2,SN30,46,years
3,SN35,61,years
4,SN36,70,years
5,SN39,57,years


## 5. Medications

In [10]:
# Extract drug ingredients and resolve patient study IDs
med_ingredients = extract_med_ingredients(med_administrations, patients)
med_ingredients

,study_id,regimen,ingredient_code,ingredient_display
0,SN1,Cemiplimab,1172911000168105,cemiplimab
1,SN10,Cisplatin/Gemcitabine/Durvalumab,21362011000036101,cisplatin
2,SN10,Cisplatin/Gemcitabine/Durvalumab,21443011000036105,gemcitabine
3,SN10,Cisplatin/Gemcitabine/Durvalumab,21836011000036104,durvalumab


In [11]:
# Look up each ingredient in AMT and find its CTPP product forms
ctpp_results = med_ingredients_to_ctpp(med_ingredients)
ctpp_results

,study_id,ingredient,ctpp_code,ctpp_display
0,SN1,Cemiplimab,1506861000168101,"Cemiplimab 350 mg/7 mL injection, vial"
1,SN1,Cemiplimab,1506871000168107,"Libtayo 350 mg/7 mL injection, 7 mL vial"
2,SN10,Cisplatin,1324391000168101,"Cisplatin (Accord) 10 mg/10 mL injection, 10 mL vial"
3,SN10,Cisplatin,1199211000168104,"Cisplatin (Accord) 100 mg/100 mL injection, 100 mL vial"
4,SN10,Cisplatin,1324461000168103,"Cisplatin (Accord) 25 mg/25 mL injection, 25 mL vial"
...,...,...,...,...
68,SN10,Gemcitabine,4932011000036105,"Gemzar 200 mg powder for injection, 200 mg vial"
69,SN10,Durvalumab,1275701000168101,"Durvalumab 120 mg/2.4 mL injection, vial"
70,SN10,Durvalumab,1275591000168100,"Durvalumab 500 mg/10 mL injection, vial"
71,SN10,Durvalumab,1275711000168103,"Imfinzi 120 mg/2.4 mL injection, 2.4 mL vial"


## 6. VCF file table per sample

Builds one row per tumour specimen, with the **biopsy site** as the sample location and
one column per VCF type (somatic SNV/indel, germline SNV/indel, somatic SV, germline SV).

The chain resolved here is:
- `DocumentReference.context.related` → `Specimen` (tumour library) → `Specimen.parent` → biopsy `Specimen.collection.bodySite`
- `DocumentReference.content[0].attachment.url` → VCF path
- `DocumentReference.category[0].coding[0].display` + `DocumentReference.description` → VCF type

In [12]:
# Build a table with one row per sample and VCF file paths as columns
vcf_table = build_vcf_table(result.bundles)
vcf_table

,subject_id,tumour_library_id,sample_location,somatic_snv_vcf,germline_snv_vcf,somatic_sv_vcf,germline_sv_vcf
0,SN1,L2200074,Right frontal lobe,../data/real/L2200074.purple.somatic.vcf.gz,../data/real/L2200074.purple.germline.vcf.gz,../data/real/L2200074.purple.sv.vcf.gz,../data/real/L2200074.purple.sv.germline.vcf.gz
1,SN10,L2201590,Liver bx,../data/real/L2201590.purple.somatic.vcf.gz,../data/real/L2201590.purple.germline.vcf.gz,../data/real/L2201590.purple.sv.vcf.gz,../data/real/L2201590.purple.sv.germline.vcf.gz
2,SN30,L2201604,Left neck lymph node,../data/real/L2201604.purple.somatic.vcf.gz,../data/real/L2201604.purple.germline.vcf.gz,../data/real/L2201604.purple.sv.vcf.gz,../data/real/L2201604.purple.sv.germline.vcf.gz
3,SN35,L2200229,Mediastinal Lymph node,../data/real/L2200229.purple.somatic.vcf.gz,../data/real/L2200229.purple.germline.vcf.gz,../data/real/L2200229.purple.sv.vcf.gz,../data/real/L2200229.purple.sv.germline.vcf.gz
4,SN36,L2200111,ureter,../data/real/L2200111.purple.somatic.vcf.gz,../data/real/L2200111.purple.germline.vcf.gz,../data/real/L2200111.purple.sv.vcf.gz,../data/real/L2200111.purple.sv.germline.vcf.gz
5,SN39,L2200443,Liver bx,../data/real/L2200443.purple.somatic.vcf.gz,../data/real/L2200443.purple.germline.vcf.gz,../data/real/L2200443.purple.sv.vcf.gz,../data/real/L2200443.purple.sv.germline.vcf.gz
